In [43]:
import typing as _t

import torch as _torch
import torch.nn as _nn


class LightWeightDecoder(_nn.Module):
    """
    Decoder module that will upsample the final patch embedding to the output dimensions.
    """

    def __init__(
            self,
            patch_embedding_operations: _nn.ModuleDict,
            fused_embedding_operations: _nn.Sequential,
            prediction_head: _nn.Module,
    ) -> None:
        super(LightWeightDecoder, self).__init__()
        self.__patch_embedding_operations = patch_embedding_operations
        self.__fused_embedding_operations = fused_embedding_operations
        self.__prediction_head = prediction_head

        self.__initialize_weights()

    @classmethod
    def create(
            cls,
            patch_embedding_scales: _t.List[_t.Tuple[int, int]],
            input_dims: _t.Tuple[int, int, int],
            output_dims: _t.Tuple[int, int, int],
            dropout_rate: float,
    ) -> 'Decoder':
        """
        Create a decoder that will upsample the final embeddings to the output dimensions.

        :param patch_embedding_scales: List of tuples containing the patch size and embedding dimension.
        :param input_dims: Input dimensions of the image.
        :param output_dims: Output dimensions of the image.
        :param dropout_rate: Dropout rate.
        :return: Decoder module.
        """
        H, W = input_dims[1:]
        num_classes = output_dims[0]
        num_scales = len(patch_embedding_scales)
        highest_resolution_patch_embedding = min(patch_embedding_scales, key=lambda x: x[0])
        out_patch_size, out_channels = highest_resolution_patch_embedding

        patch_embedding_scales.remove(highest_resolution_patch_embedding)

        patch_embedding_operations = _nn.ModuleDict(
            {
                f'x{i}': _nn.Sequential(
                    _nn.Conv2d(in_channels=embed_dim, out_channels=out_channels, kernel_size=1, stride=1),
                    _nn.BatchNorm2d(out_channels),
                    _nn.ReLU(),
                    _nn.Dropout(p=dropout_rate),
                    _nn.Upsample(scale_factor=patch_size / out_patch_size, mode='nearest'),
                )
                for i, (patch_size, embed_dim) in enumerate(patch_embedding_scales, start=1)
            }
        )
        fused_embedding_operations = _nn.Sequential(
            _nn.Conv2d(in_channels=out_channels * num_scales, out_channels=out_channels, kernel_size=1, stride=1),
            _nn.BatchNorm2d(out_channels),
            _nn.ReLU(),
            _nn.Dropout(p=dropout_rate),
            _nn.Upsample(
                scale_factor=out_patch_size,
                mode='nearest',
            ),
        )
        prediction_head = _nn.Conv2d(in_channels=out_channels, out_channels=num_classes, kernel_size=1, stride=1)

        return cls(
            patch_embedding_operations=patch_embedding_operations,
            fused_embedding_operations=fused_embedding_operations,
            prediction_head=prediction_head
        )

    @property
    def prediction_head(self) -> _nn.Module:
        """
        Get the prediction head of the decoder

        :return: Prediction head.
        """
        return self.__prediction_head

    def forward(
            self,
            patch_embeddings: _t.Dict[str, _torch.Tensor],
            apply_prediction_head: bool = True,
    ) -> _torch.Tensor:
        """
        Forward pass of the decoder to up sample and apply prediction head to a patch embedding tensor.

        :param patch_embeddings: Patch embeddings to up sample.
        :param apply_prediction_head: Whether to apply the prediction head.
        :return: Predicted output tensor.
        """
        patch_embedding_operations = self.__patch_embedding_operations
        fused_embedding_operations = self.__fused_embedding_operations
        prediction_head = self.__prediction_head

        key = set(
            patch_embeddings.keys()).difference(
            set(patch_embedding_operations.keys())
        ).pop()
        patch_embedding = patch_embeddings[key]
        B, P, E = patch_embedding.shape
        P = int(P ** 0.5)
        patch_embedding = patch_embedding.reshape(B, E, P, P)

        # Apply the patch embedding operations
        feature_maps = [patch_embedding, ]
        for key, operation in patch_embedding_operations.items():
            patch_embedding = patch_embeddings[key]
            B, P, E = patch_embedding.shape
            P = int(P ** 0.5)
            patch_embedding = patch_embedding.reshape(B, E, P, P)
            feature_maps.append(operation(patch_embedding))

        # Fuse the patch embeddings
        x = fused_embedding_operations(_torch.cat(feature_maps, dim=1))

        # Apply the prediction head
        if apply_prediction_head:
            x = prediction_head(x)

        return x

    def __initialize_weights(self) -> None:
        """
        Initialize the weights of the decoder.
        """
        for module in self.modules():
            if isinstance(module, _nn.Conv2d):
                _nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
                if module.bias is not None:
                    _nn.init.constant_(module.bias, 0)
            elif isinstance(module, _nn.BatchNorm2d):
                _nn.init.constant_(module.weight, 1)
                _nn.init.constant_(module.bias, 0)




In [44]:
import torch as _torch

scale_1 = _torch.rand(1, 1024, 768)
scale_2 = _torch.rand(1, 4096, 512)

inputs = {
    'x1': scale_1,
    'x2': scale_2
}

decoder = LightWeightDecoder.create(
    patch_embedding_scales=[(16, 768), (8, 512)],
    input_dims=(3, 512, 512),
    output_dims=(1, 512, 512),
    dropout_rate=0.1
)

output = decoder(inputs)
output.shape

torch.Size([1, 1, 512, 512])

In [45]:
output

tensor([[[[-60.1547, -60.1547, -60.1547,  ..., -23.6192, -23.6192, -23.6192],
          [-60.1547, -60.1547, -60.1547,  ..., -23.6192, -23.6192, -23.6192],
          [-60.1547, -60.1547, -60.1547,  ..., -23.6192, -23.6192, -23.6192],
          ...,
          [ -9.7661,  -9.7661,  -9.7661,  ..., -33.9507, -33.9507, -33.9507],
          [ -9.7661,  -9.7661,  -9.7661,  ..., -33.9507, -33.9507, -33.9507],
          [ -9.7661,  -9.7661,  -9.7661,  ..., -33.9507, -33.9507, -33.9507]]]],
       grad_fn=<ConvolutionBackward0>)

In [4]:
import torch as _torch
import torch.nn as _nn
import typing as _t


class PatchFusion(_nn.Module):
    """
    Patch fusion layer that will fuse the embeddings of the patches to a common scale.
    """

    def __init__(
            self, *, in_dims: _t.List[_t.List[int]], out_patches: int, out_embed: int, dropout_rate: float
    ) -> None:
        """

        :param in_dims: The dimensions of the input tensors.
        :param out_patches: The number of output patches.
        :param out_embed: The length of the output patch embeddings.
        :param dropout_rate: Dropout rate.
        """
        super(PatchFusion, self).__init__()

        out_resolution = int(out_patches ** 0.5)
        in_resolutions = []
        patch_embedding_projectors = _nn.ModuleList()
        for in_patches, in_embed in in_dims:
            in_resolution = int(in_patches ** 0.5)
            if in_patches < out_patches:
                scale = out_resolution // in_resolution
                operation = _nn.Sequential(
                    _nn.Conv2d(
                        in_channels=in_embed, out_channels=out_embed, kernel_size=1, stride=1
                    ),
                    _nn.Upsample(scale_factor=scale, mode="nearest")
                )
            elif in_patches > out_patches:
                scale = in_resolution // out_resolution
                operation = _nn.Conv2d(
                    in_channels=in_embed, out_channels=out_embed, kernel_size=scale, stride=scale
                )
            else:
                operation = _nn.Identity()

            patch_embedding_projectors.append(_nn.Sequential(
                operation,
                _nn.BatchNorm2d(out_embed),
                _nn.ReLU(),
                _nn.Dropout(dropout_rate)
            ))
            in_resolutions.append(in_resolution)

        self.__patch_embedding_projectors = patch_embedding_projectors
        self.__out_resolution = out_resolution
        self.__in_resolutions = in_resolutions
        self.__out_patches = out_patches
        self.__out_embed = out_embed
        self.__initialize_weights()

    def forward(self, target_tensor: _torch.Tensor, tensors: _t.List[_torch.Tensor]) -> _torch.Tensor:
        """
        Forward pass of the patch fusion layer to merge together a given tensor with a target tensor by modifying
        spatial dimension using learnable operations.

        :param target_tensor: Target tensor to be fused with. Shape (batch_size, out_patches, out_embed).
        :param tensors: List of tensors to be fused. Shape (batch_size, in_patches, in_embed).
        :return: Fused tensor. Shape (batch_size, out_patches, out_embed).
        """
        patch_embedding_projectors = self.__patch_embedding_projectors
        in_resolutions = self.__in_resolutions
        out_resolution = self.__out_resolution
        out_patches = self.__out_patches
        out_embed = self.__out_embed

        # Reshape the target tensor to (B, E, P, P)
        B, P, E = target_tensor.shape
        target_tensor = target_tensor.reshape(B, out_resolution, out_resolution, E).permute(0, 3, 1, 2).contiguous()

        # Apply the patch embedding projectors
        for tensor, projector, in_resolution in zip(tensors, patch_embedding_projectors, in_resolutions):
            B, P, E = tensor.shape
            tensor = tensor.reshape(B, in_resolution, in_resolution, E).permute(0, 3, 1, 2).contiguous()
            tensor = projector(tensor)
            target_tensor = target_tensor + tensor

        target_tensor = target_tensor.permute(0, 3, 1, 2).reshape(B, out_patches, out_embed).float()

        return target_tensor

    def __initialize_weights(self) -> None:
        """
        Initialize the weights of the patch fusion layer.
        """
        patch_embedding_projectors = self.__patch_embedding_projectors

        for projector in patch_embedding_projectors:
            for module in projector.modules():
                if isinstance(module, _nn.Conv2d):
                    _nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='relu')
                    if module.bias is not None:
                        _nn.init.constant_(module.bias, 0)
                elif isinstance(module, _nn.BatchNorm2d):
                    _nn.init.constant_(module.weight, 1)
                    _nn.init.constant_(module.bias, 0)

In [5]:
import torch as _torch

patch_fusion = PatchFusion(
    in_dims=[[1024, 768], [4096, 512]],
    out_patches=256,
    out_embed=1024,
    dropout_rate=0.25,
)

target = _torch.randn(10, 256, 1024)
x1 = _torch.randn(10, 1024, 768)
x2 = _torch.randn(10, 4096, 512)

y = patch_fusion(target_tensor=target, tensors=[x1, x2])
y.shape

torch.Size([10, 256, 1024])